# GAE_HEP on Kaggle GPU: test-split confirmation of one configuration

Trains one configuration of the energy-flow graph VAE with five seeds under the development protocol and scores every seed on the test splits (`gaehep dev` -> `gaehep promote`), then builds the seed ensemble, its mass-decorrelated variant, the bump hunt and the weak-supervision study, and zips the results (`/kaggle/working/gaehep_confirm.zip`).  Default: `configs/ef_vae_gpu_n100.yaml`, the GPU grid winner with 100 constituents per jet (the one scale variant of the GPU study that improved the 2-prong dev AUC).

> **Before you run:** switch **Internet on** and select a **GPU accelerator** in the session options, then *Save Version -> Save & Run All (Commit)*: about 1-1.5 h.

In [ ]:
!nvidia-smi

import os
import time

# notebook start time, for the time guards of the study cell (Kaggle stops a session after 12 h)
os.environ.setdefault("GAEHEP_NB_T0", str(time.time()))

REPO_URL = "https://github.com/raktim711/GAE_HEP.git"
REPO_DIR = "/kaggle/working/GAE_HEP"

# clone once; skipped when the repo is already there (e.g. after a kernel restart)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

import torch

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU visible - choose a GPU accelerator in the session options.")

In [ ]:
import os
import shutil

# Results go to /kaggle/working (saved with the notebook version).  The ~6 GB of data go to /tmp when there is
# room (not saved as notebook output, which is limited to 20 GB), otherwise to /kaggle/working/data.
DATA_DIR = "/tmp/gaehep_data"
if not (os.path.isdir(DATA_DIR) or shutil.disk_usage("/tmp").free > 20e9):
    DATA_DIR = "/kaggle/working/data"
os.environ["GAEHEP_DATA"] = DATA_DIR
os.environ["GAEHEP_RESULTS"] = "/kaggle/working/results"
print("data ->", DATA_DIR, "| free disk in /tmp:", round(shutil.disk_usage("/tmp").free / 1e9), "GB")

# data pipeline; the shell commands inherit os.environ.
# `download` is md5-verified and resumable: re-run this cell if it gets interrupted.
!gaehep download
!gaehep preprocess --sample rnd --workers 4
!gaehep preprocess --sample qqq --workers 4
!gaehep tables --sample rnd
!gaehep tables --sample qqq
!gaehep splits
!gaehep baselines

In [ ]:
import os
import shlex
import subprocess
import time
import zipfile
from pathlib import Path

REPO_DIR = "/kaggle/working/GAE_HEP"
RESULTS = Path("/kaggle/working/results")
ZIP_PATH = Path("/kaggle/working/gaehep_confirm.zip")
os.environ["GAEHEP_DATA"] = "/tmp/gaehep_data" if os.path.isdir("/tmp/gaehep_data") else "/kaggle/working/data"
os.environ["GAEHEP_RESULTS"] = str(RESULTS)
os.chdir(REPO_DIR)
T_START = float(os.environ.get("GAEHEP_NB_T0", time.time()))

CONFIG = "configs/ef_vae_gpu_n100.yaml"
NAME = "n100"  # runs: results/dev/<NAME>_dev_s<s>, results/runs/<NAME>_s<s>, results/runs/<NAME>_seeds
SEEDS = [1, 2, 3, 4, 5]
# `gaehep dev` resets the train section to the 5-epoch dev budget: the configuration's own training settings
TRAIN = {"train.epochs": 2, "train.jets_per_epoch": 120000, "train.device": "cuda", "train.threads": 4}


def hours():
    return (time.time() - T_START) / 3600


def gaehep(*args, check=True):
    cmd = ["gaehep", *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, check=check, cwd=REPO_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    print(f"  ({(time.time() - t0) / 60:.1f} min, notebook at {hours():.2f} h)", flush=True)
    return r.returncode


def zip_results():
    n = 0
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for f in RESULTS.rglob("*"):
            if f.is_file() and f.suffix not in {".pt", ".ckpt"} and "checkpoint" not in f.name and f.stat().st_size < 50e6:
                z.write(f, f.relative_to(RESULTS.parent))
                n += 1
    print(f"zipped {n} files -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)


sets = [x for k, v in TRAIN.items() for x in ("--set", f"{k}={v}")]
runs = []
for s in SEEDS:
    if hours() > 10.5:
        break
    dev_name, run_name = f"{NAME}_dev_s{s}", f"{NAME}_s{s}"
    gaehep("dev", CONFIG, "--name", dev_name, *sets, "--set", f"train.seed={s}")
    gaehep("promote", dev_name, run_name)
    runs.append(run_name)
gaehep("ensemble", *runs, "--out", f"{NAME}_seeds")
gaehep("qn", f"{NAME}_seeds", check=False)
gaehep("bumphunt", f"{NAME}_seeds", "--agg", "prod", check=False)
gaehep("weak", runs[0], "--n-models", "5", check=False)
gaehep("report", check=False)
gaehep("dev-report", check=False)
zip_results()